In [ ]:
import os, sys
import torch
import argparse
import numpy as np
import torch.utils.data
from easydict import EasyDict as edict
from timeit import default_timer as timer
from tqdm import tqdm

from utils.eval import Metric
from utils.gpu_dispatch import GPU
from utils.common_utils import dir_check, to_device, ws, unfold_dict, dict_merge, GpuId2CudaId, Logger

from algorithm.dataset import CleanDataset, TrafficDataset
from algorithm.diffstg.model import DiffSTG, save2file

import matplotlib.pyplot as plt

In [ ]:
trained_model_path = './output/model/ewz_preprocessed_1day_1hour_50epoch.dm4stg'

DATA_path = './data/dataset/EWZ_preprocessed/'
flow_path = os.path.join(DATA_path, 'flow.npy')
adj_path = os.path.join(DATA_path, 'adj.npy')

In [ ]:
flow = np.load(flow_path)
adj = np.load(adj_path) 

flow.shape

In [ ]:
T = flow.shape[0]
sensor_idx = 2
plt.plot(range(T), flow[:,sensor_idx,0])

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torch.load(trained_model_path, map_location=device, weights_only=False)

In [ ]:
y_true = np.load('./true.npy')
y_pred = np.load('./pred.npy')

y_true.shape, y_pred.shape

In [ ]:


horizon_idx = 0  # 1-step ahead horizon
num_nodes_to_plot = 50

fig, axes = plt.subplots(num_nodes_to_plot, 1, figsize=(10, 2.5 * num_nodes_to_plot), sharex=True)

for i in range(num_nodes_to_plot):
    node_true = y_true[:, horizon_idx, i, 0]
    node_pred = y_pred[:, 0, horizon_idx, i, 0]
    
    ax = axes[i]
    ax.plot(node_true, label='True')
    ax.plot(node_pred, label='Predicted', linestyle='--', alpha=0.7)
    ax.set_title(f'Node {i}, Horizon {horizon_idx+1}')
    ax.grid(True)
    ax.legend(loc='upper right')


    ax.set_ylim(0., np.sort(node_true)[-3])
    
    if i == num_nodes_to_plot - 1:
        ax.set_xlabel('Time Index')
    ax.set_ylabel('Value')
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.savefig('ewz_preprocessed_preds.pdf')
plt.show()
